# 12. Umbral operativo del programa de tamizacion — Fase 6

Modelo canonico **XGBoost-sin-avicena** (58 feats, regional conservada). Asignacion elegida:
**estratificada por regional**. Aqui se elige el **punto de corte** (cuantas mujeres tamizar).

Para cada fraccion top-k% se reporta:
- **cupos**: leads seleccionados (mujeres a tamizar).
- **umbral_prob**: probabilidad minima del corte.
- **recall**: % de casos incidentes reales captados.
- **VPP (precision)**: % de leads que son caso real.
- **lift**: VPP / tasa base (cuanto mejor que tamizar al azar).
- **NNS**: number needed to screen = 1/VPP (mujeres a tamizar por cada caso hallado).

> Tasa base ~0,04% => VPP bajo y NNS alto por naturaleza (incidencia, no diagnostico).
> El valor del modelo es el **lift**: concentra los casos en una fraccion pequena.


In [1]:
import json, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
import joblib
from sklearn.metrics import average_precision_score

B = "bases"
model = joblib.load(f"{B}/modelo_fase6_XGBoost-sin-avicena.joblib")
nat_va = pd.read_parquet(f"{B}/prediccion_mama_val_native.parquet")
FEAT = [c for c in nat_va.columns if c not in ('key', 'label', 'tiene_avicena')]
assert model.n_features_in_ == len(FEAT), (model.n_features_in_, len(FEAT))
y = nat_va['label'].values.astype(int)
X = nat_va[FEAT].astype('float32').values
p = model.predict_proba(X)[:, 1]
N, P = len(y), int(y.sum())
base = P / N
REG = [c for c in nat_va.columns if c.startswith('REG_')]
reg_names = [c.replace('REG_', '').strip() for c in REG]
regional = np.array([reg_names[i] for i in nat_va[REG].values.argmax(axis=1)])
print(f"Validacion 2023->2024: N={N:,} | positivos={P} | tasa base={base*100:.4f}% | AUC-PR={average_precision_score(y,p):.4f}")


Validacion 2023->2024: N=2,410,807 | positivos=993 | tasa base=0.0412% | AUC-PR=0.0388


## 1. Curva de operacion — GLOBAL (top-k% absoluto)

In [2]:
def sel_global(frac):
    k = max(1, int(N * frac))
    s = np.zeros(N, dtype=bool); s[np.argpartition(-p, k)[:k]] = True
    return s

def sel_strat(frac):
    s = np.zeros(N, dtype=bool)
    for r in reg_names:
        idx = np.where(regional == r)[0]
        kk = max(1, int(len(idx) * frac))
        s[idx[np.argpartition(-p[idx], kk)[:kk]]] = True
    return s

def fila(s, frac):
    n = int(s.sum()); tp = int(y[s].sum())
    vpp = tp / n if n else 0
    return {'top_k%': frac*100, 'cupos': n, 'umbral_prob': float(p[s].min()),
            'recall': tp / P, 'VPP%': vpp*100, 'lift': (vpp/base) if base else 0,
            'NNS': (1/vpp) if vpp else np.inf}

FRACS = [0.005, 0.01, 0.02, 0.03, 0.05, 0.10, 0.15, 0.20]
tab_g = pd.DataFrame([fila(sel_global(f), f) for f in FRACS])
pd.set_option('display.width', 200, 'display.max_columns', 20)
print("=== GLOBAL (top-k% absoluto) ===")
print(tab_g.to_string(index=False, float_format='{:.3f}'.format))


=== GLOBAL (top-k% absoluto) ===
 top_k%  cupos  umbral_prob  recall  VPP%   lift     NNS
  0.500  12054        0.026   0.274 2.257 54.784  44.316
  1.000  24108        0.017   0.333 1.373 33.333  72.834
  2.000  48216        0.010   0.416 0.857 20.796 116.746
  3.000  72324        0.008   0.483 0.664 16.113 150.675
  5.000 120540        0.005   0.568 0.468 11.360 213.723
 10.000 241080        0.003   0.699 0.288  6.989 347.378
 15.000 361621        0.002   0.773 0.212  5.156 470.861
 20.000 482161        0.001   0.824 0.170  4.119 589.439


## 2. Curva de operacion — ESTRATIFICADO por regional (asignacion elegida)

In [3]:
tab_s = pd.DataFrame([fila(sel_strat(f), f) for f in FRACS])
print("=== ESTRATIFICADO (top-k% por regional) ===")
print(tab_s.to_string(index=False, float_format='{:.3f}'.format))
print("\nNota: recall estratificado ~1pp por debajo del global; cupos casi iguales; VPP/NNS similares.")


=== ESTRATIFICADO (top-k% por regional) ===
 top_k%  cupos  umbral_prob  recall  VPP%   lift     NNS
  0.500  12052        0.015   0.263 2.166 52.577  46.176
  1.000  24105        0.010   0.329 1.357 32.935  73.716
  2.000  48214        0.006   0.412 0.848 20.595 117.883
  3.000  72321        0.005   0.474 0.651 15.811 153.548
  5.000 120538        0.003   0.556 0.458 11.118 218.366
 10.000 241078        0.002   0.693 0.285  6.929 350.404
 15.000 361618        0.001   0.754 0.207  5.029 482.801
 20.000 482160        0.001   0.806 0.166  4.028 602.700

Nota: recall estratificado ~1pp por debajo del global; cupos casi iguales; VPP/NNS similares.


## 3. Escalado a poblacion de produccion 2024

La validacion (2023->2024) aproxima el tamano de produccion. Se muestran los cupos absolutos
que implicaria cada corte para dimensionar la capacidad del programa.


In [4]:
esc = tab_s[['top_k%', 'cupos', 'recall', 'VPP%', 'lift', 'NNS']].copy()
esc['casos_esperados_captados'] = (esc['recall'] * P).round(0).astype(int)
esc['casos_totales'] = P
print("=== Dimensionamiento (estratificado) ===")
print(esc.to_string(index=False, float_format='{:.2f}'.format))
print(f"\nLectura: con top-5% se tamizan ~{int(N*0.05):,} mujeres y se captan "
      f"~{int(tab_s.iloc[4]['recall']*P)} de {P} casos ({tab_s.iloc[4]['recall']*100:.0f}%).")


=== Dimensionamiento (estratificado) ===
 top_k%  cupos  recall  VPP%  lift    NNS  casos_esperados_captados  casos_totales
   0.50  12052    0.26  2.17 52.58  46.18                       261            993
   1.00  24105    0.33  1.36 32.93  73.72                       327            993
   2.00  48214    0.41  0.85 20.60 117.88                       409            993
   3.00  72321    0.47  0.65 15.81 153.55                       471            993
   5.00 120538    0.56  0.46 11.12 218.37                       552            993
  10.00 241078    0.69  0.29  6.93 350.40                       688            993
  15.00 361618    0.75  0.21  5.03 482.80                       749            993
  20.00 482160    0.81  0.17  4.03 602.70                       800            993

Lectura: con top-5% se tamizan ~120,540 mujeres y se captan ~552 de 993 casos (56%).


## 4. Guardar curva de operacion

In [5]:
out = {
    'modelo': 'XGBoost-sin-avicena (estratificado)',
    'N_val': N, 'positivos': P, 'tasa_base': base,
    'curva_global': tab_g.replace([np.inf], None).to_dict(orient='records'),
    'curva_estratificada': tab_s.replace([np.inf], None).to_dict(orient='records'),
}
with open(f"{B}/umbral_operativo.json", "w", encoding="utf-8") as f:
    json.dump(out, f, indent=2, ensure_ascii=False, default=float)
print("Guardado: bases/umbral_operativo.json")
print("\nElegir corte segun capacidad del programa (cupos/ano). Tabla lista para decidir.")


Guardado: bases/umbral_operativo.json

Elegir corte segun capacidad del programa (cupos/ano). Tabla lista para decidir.
